# PT-W3-D3 概念实验：Ontology 约束 Agent 认知

Bounded Context 是认知围栏；D-014 保证业务事实只有一个 Owner；Context Map 规定跨域协作通道。

## 实验 1：边界内回答，越界则拒绝或走受控引用

模拟一个 Lease Agent：能理解租约和铺位可用性，但不能把商户、财务事实复制成自己的真值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

CONTEXTS = {
    'Lease/Occupancy': {'entities': {'Lease', 'Space'}, 'capabilities': {'check_availability'}, 'owner_facts': {'occupancy_state'}},
    'Contract': {'entities': {'Contract'}, 'capabilities': {'read_clause'}, 'owner_facts': {'contract_clause'}},
    'Merchant': {'entities': {'Tenant'}, 'capabilities': {'read_merchant'}, 'owner_facts': {'merchant_identity'}},
    'Billing': {'entities': {'Bill'}, 'capabilities': {'calculate_bill'}, 'owner_facts': {'bill_amount'}},
}
CONTEXT_MAP = {('Lease/Occupancy', 'Contract'): 'upstream_reference', ('Lease/Occupancy', 'Merchant'): 'shared_identity_reference'}
print('Lease Agent 认知范围:', CONTEXTS['Lease/Occupancy'])

In [ ]:
def route_query(agent_context, entity, action):
    local = CONTEXTS[agent_context]
    if entity in local['entities'] and action in local['capabilities']:
        return 'allow: 本 Context 自有能力'
    for target, spec in CONTEXTS.items():
        if entity in spec['entities'] and (agent_context, target) in CONTEXT_MAP:
            return f'reference: 通过 Context Map -> {target}，只消费 Owner 事实'
    return 'deny: 越过认知边界，未声明协作通道'

requests = [('Lease', 'check_availability'), ('Contract', 'read_clause'), ('Bill', 'calculate_bill'), ('Tenant', 'delete_identity')]
for req in requests: print(req, '=>', route_query('Lease/Occupancy', *req))

In [ ]:
outcomes = [route_query('Lease/Occupancy', *r) for r in requests]
allow = [int(o.startswith('allow')) for o in outcomes]
reference = [int(o.startswith('reference')) for o in outcomes]
deny = [int(o.startswith('deny')) for o in outcomes]
print('allow/reference/deny =', sum(allow), sum(reference), sum(deny))
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['本域允许', '受控引用', '越界拒绝'], [sum(allow), sum(reference), sum(deny)], color=['#2a9d8f', '#e9c46a', '#e76f51'])
ax.set_title('Bounded Context 认知围栏'); ax.set_ylabel('请求数')
plt.tight_layout(); plt.show()